In [1]:
from sqlalchemy import create_engine
import pandas as pd
import plotly.express as px

PROJECT_ID = "sg-20min-town-mod2proj"
engine = create_engine(f"bigquery://{PROJECT_ID}/olist_marts")
print("Connected!")

Connected!


In [2]:
sql = """
    SELECT
        rfm_segment,
        COUNT(*)                    AS customer_count,
        ROUND(AVG(monetary), 2)     AS avg_ltv,
        ROUND(SUM(monetary), 2)     AS total_revenue,
        ROUND(AVG(recency_days), 0) AS avg_recency_days,
        ROUND(AVG(frequency), 2)    AS avg_frequency
    FROM olist_marts.mart_customer_rfm
    GROUP BY rfm_segment
    ORDER BY avg_ltv DESC
"""
df_seg = pd.read_sql(sql, engine)
df_seg["revenue_pct"] = (
    df_seg["total_revenue"] / df_seg["total_revenue"].sum() * 100
).round(1)
print(df_seg)

   rfm_segment  customer_count  avg_ltv  total_revenue  avg_recency_days  \
0          VIP              34  1000.10       34003.33             123.0   
1  Big Spender            3896   928.26     3616496.75             300.0   
2        Loyal            1979   301.92      597491.67             213.0   
3      At-Risk           27949   129.99     3633027.42             489.0   
4    One-Timer           59500   126.58     7531579.66             215.0   

   avg_frequency  revenue_pct  
0           4.12          0.2  
1           1.00         23.5  
2           2.09          3.9  
3           1.03         23.6  
4           1.00         48.9  


In [3]:
colors = {
    "VIP":"#27ae60","Loyal":"#2980b9",
    "Big Spender":"#f39c12","At-Risk":"#e74c3c","One-Timer":"#95a5a6"
}

fig1 = px.pie(df_seg, names="rfm_segment", values="customer_count",
    title="Customer Segment Distribution", color_discrete_map=colors)
fig1.show()
fig1.write_image("../docs/chart_03_rfm_pie.png", width=900, height=600)

fig2 = px.bar(df_seg.sort_values("total_revenue", ascending=False),
    x="rfm_segment", y="total_revenue",
    title="Revenue by Segment", color="rfm_segment",
    color_discrete_map=colors, text="revenue_pct")
fig2.update_traces(texttemplate="%{text}%", textposition="outside")
fig2.show()
fig2.write_image("../docs/chart_03_rfm_revenue.png", width=1000, height=600)
print("Saved RFM charts")

Saved RFM charts


In [4]:
sql2 = """
    SELECT customer_unique_id, recency_days, frequency, monetary, rfm_segment
    FROM olist_marts.mart_customer_rfm WHERE monetary < 5000 LIMIT 5000
"""
df_ind = pd.read_sql(sql2, engine)
fig3 = px.scatter(df_ind, x="recency_days", y="frequency",
    color="rfm_segment", size="monetary",
    title="RFM Scatter: Recency vs Frequency (size = spend)",
    color_discrete_map=colors)
fig3.show()
fig3.write_image("../docs/chart_03_rfm_scatter.png", width=1200, height=600)
print("Saved RFM scatter")

Saved RFM scatter
